# Use the `copilot-ml-trainer` hosted agent from Python

Talks to the Foundry **hosted** agent with the `azure-ai-projects` SDK and a Microsoft Entra
token (no API keys). The SDK gives you an OpenAI client routed at the agent plus typed helpers
for sessions and session files.

1. Auth + client
2. One-shot ask
3. Multi-turn conversation
4. Streaming
5. Train a model from your own data (upload the dataset to the session's files)
6. Download the trained model (`model.pkl` **and** the full MLflow model)

**How data reaches the agent:** a hosted agent gets a per-session sandbox with a real
filesystem. You create a session, upload your dataset to it, then bind each call to that
session with `agent_session_id`. The agent reads the file, trains, and writes artifacts back
to the sandbox, which you download with `download_session_file`.

In [ ]:
# On a Microsoft-managed device, use the internal package proxy:
# %pip install --index-url https://packagefeedproxy.microsoft.io/pypi/simple "azure-ai-projects>=2.3.0" azure-identity openai pandas joblib scikit-learn python-dotenv
%pip install "azure-ai-projects>=2.3.0" azure-identity openai pandas joblib scikit-learn python-dotenv

In [26]:
import os

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Connection settings live in .env (see .env.example). Populate it from
# `azd env get-values` in this folder: AZURE_AI_PROJECT_ENDPOINT -> PROJECT_ENDPOINT,
# AGENT_COPILOT_ML_TRAINER_NAME -> AGENT_NAME.
load_dotenv()
PROJECT = os.environ["PROJECT_ENDPOINT"]
AGENT = os.environ["AGENT_NAME"]

# `az login` (or VS Code / managed identity) backs the credential.
project = AIProjectClient(endpoint=PROJECT, credential=DefaultAzureCredential())

# An OpenAI client already routed at this agent's Responses endpoint.
agent = project.get_openai_client(agent_name=AGENT)

## 1. One-shot ask
First call is a cold start (the micro-VM spins up), so expect a few seconds of latency.

In [27]:
resp = agent.responses.create(input="In one sentence, what can you help me with?")
print(resp.output_text)

I can help with software engineering tasks like reading and editing code, debugging errors, running tests/builds, reviewing changes, configuring environments, and training/exporting ML models from tabular datasets.


## 2. Multi-turn conversation
Pass the previous response's `id` as `previous_response_id`. History is stored server-side;
you only carry the id.

In [28]:
r1 = agent.responses.create(input="My favorite ML task is binary classification. Remember that.")
print("A:", r1.output_text)

r2 = agent.responses.create(input="What did I say my favorite task was?", previous_response_id=r1.id)
print("B:", r2.output_text)

A: Got it — your favorite ML task is **binary classification**.
B: Your favorite ML task is **binary classification**.


## 3. Streaming
Stream tokens as they are produced.

In [29]:
with agent.responses.stream(model=AGENT, input="List three common classification metrics, one per line.") as stream:
    for event in stream:
        if event.type == "response.output_text.delta":
            print(event.delta, end="", flush=True)
    print()

Accuracy  
Precision  
Recall


## 4. Train a model from your own data
Make a first call to spin up a session, capture its `agent_session_id`, upload the dataset
into that session's sandbox, then bind the training call to the same session. Upload paths are
relative to `$HOME` (`/home/session`), so `data/input.csv` lands at `$HOME/data/input.csv`
inside the container.

In [30]:
import pandas as pd

# Replace with your own data, e.g. open("data/mydata.csv", "rb").read().
df = pd.DataFrame({
    "a": [1, 3, 5, 7, 2, 8, 1, 9, 4, 6],
    "b": [2, 4, 6, 8, 1, 9, 1, 9, 3, 7],
    "label": [0, 1, 0, 1, 0, 1, 0, 1, 0, 1],
})

# Create a session by making a first call, then capture its id from model_extra.
seed = agent.responses.create(input="Session ready - I'll upload a dataset next.")
session_id = seed.model_extra["agent_session_id"]
print("session:", session_id)

# Upload the dataset to $HOME/data/input.csv.
project.agents.upload_session_file(
    agent_name=AGENT,
    session_id=session_id,
    content=df.to_csv(index=False).encode(),
    path="data/input.csv",
)

# Train, binding the call to the session so the agent can read the uploaded file.
train = agent.responses.create(
    input="Read $HOME/data/input.csv (target column = 'label'). Train the best classifier you "
          "can with a train/test split. Save model.pkl and metrics.json under $HOME, log the "
          "model with MLflow to $HOME/mlflow_model, and zip that folder into "
          "$HOME/mlflow_model.zip. Print a short summary of the model and its metrics.",
    extra_body={"agent_session_id": session_id},
)
print(train.output_text)

session: 027062429b9f04700ace0702bb573b77734267a98d6bdd4b547a839987ec47d
I’ll load the dataset, infer suitable preprocessing/classifiers, train with a held-out split, then export the requested artifacts under `$HOME`.The dataset is very small and balanced, so I’ll compare a few robust scikit-learn classifiers on the training split and export the best evaluated pipeline.MLflow rejected the selected tree-based model under its safe serialization checks, so I’m rerunning the export with explicit trust for the model types produced locally during training.Trained and exported the classifier artifacts under `/home/session`.

**Model:** scikit-learn `RandomForestClassifier` pipeline  
**Task:** classification  
**Target:** `label`  
**Metrics:** accuracy `0.50`, macro F1 `0.3333`, ROC AUC `0.75`

Created:
- `/home/session/model.pkl`
- `/home/session/metrics.json`
- `/home/session/mlflow_model/`
- `/home/session/mlflow_model.zip`


## 5. Retrieve the artifacts
The agent wrote everything into the session sandbox, so download it with
`download_session_file` (it yields byte chunks). Paths are relative to `$HOME`. The MLflow
model is a *directory*, so we pull the `mlflow_model.zip` the agent made and unpack it — that
gives you the model plus its `MLmodel`, `conda.yaml`, and `requirements.txt` so it carries its
own environment.

In [31]:
import zipfile


def download(path: str, dest: str) -> int:
    data = b"".join(project.agents.download_session_file(
        agent_name=AGENT, session_id=session_id, path=path))
    with open(dest, "wb") as f:
        f.write(data)
    return len(data)


print(f"model.pkl      {download('model.pkl', 'model.pkl'):>8} bytes")
print(f"metrics.json   {download('metrics.json', 'metrics.json'):>8} bytes")
print(f"mlflow_model.zip {download('mlflow_model.zip', 'mlflow_model.zip'):>6} bytes")

# Unpack the MLflow model directory.
with zipfile.ZipFile("mlflow_model.zip") as z:
    z.extractall("mlflow_model")
print("extracted ->", sorted(p.name for p in __import__("pathlib").Path("mlflow_model").rglob("*")))

model.pkl        127947 bytes
metrics.json       1607 bytes
mlflow_model.zip 107140 bytes
extracted -> ['MLmodel', 'conda.yaml', 'input_example.json', 'mlflow_model', 'model.pkl', 'model.skops', 'python_env.yaml', 'requirements.txt', 'serving_input_example.json']


### Load the model locally
Two options. Load the raw estimator from `model.pkl` (needs a matching scikit-learn version,
1.9.x), or load the MLflow model, which carries its own environment spec so it's more portable.

In [33]:
import joblib
import pandas as pd

model = joblib.load("model.pkl")
print(type(model).__name__)

# The pipeline remembers the columns it was trained on:
print("expected columns:", list(model.feature_names_in_))

X = pd.DataFrame([[1, 2], [8, 9]], columns=model.feature_names_in_)
print("prediction:", model.predict(X))

Pipeline
expected columns: ['a', 'b']
prediction: [0 1]
